In [ ]:
import pickle
with open('dpp_error.pk', 'rb') as f:
    dpp_error = pickle.load(f)
    
from dppy.finite_dpps import FiniteDPP
import torch
    

In [ ]:
(vec_matrix, encs, probs, tactics, state, theorem, temperature) = dpp_error


In [ ]:
temperature = 1
scale = 1

logprobs = [t[1] / temperature for t in tactics]

probs = torch.softmax(torch.tensor(logprobs), dim=0) * scale

sim_matrix_ = torch.cat(encs[:16], dim=0)


sim_matrix = sim_matrix_ @ sim_matrix_.t()
sim_matrix = sim_matrix.cpu().numpy()


vec_matrix = torch.mul(sim_matrix_.cpu(), probs[:16].unsqueeze(1)).numpy()
vec_matrix = vec_matrix @ vec_matrix.T


In [ ]:

DPP = FiniteDPP('likelihood', **{'L': vec_matrix})
# DPP = FiniteDPP('likelihood', **{'L': vec_matrix})
DPP.compute_K()

k_sum = sum(DPP.K_eig_vals)
print(k_sum)

DPP.sample_exact_k_dpp(size=8, mode='KuTa12')#, rng
# DPP.sample_exact()#, rng

len(DPP.list_of_samples[0])

In [ ]:
filtered = [tactics[i][0] for i in sorted(DPP.list_of_samples[0])]
filtered

In [ ]:
[t[0] for t in tactics[:16] if t[0] not in filtered]

In [ ]:
sim_matrix

In [ ]:
[(i,t[0], probs[i].item()) for i,t in enumerate(tactics[:16])]

In [ ]:
samples = []
for _ in range(8):
    DPP = FiniteDPP('likelihood', **{'L': vec_matrix})
    DPP.sample_exact_k_dpp(size=3, mode='KuTa12')#, rng
    samples.append(DPP.list_of_samples[0])

from collections import Counter
Counter([tactics[i][0]  for s in samples for i in sorted(s)])


In [ ]:

# plot the co-occurences of tactics, pairwise
co_occurences = torch.zeros((8, 8))
for s in samples:
    for i in s:
        for j in s:
            co_occurences[i,j] += 1
            
co_occurences = co_occurences / len(samples)

import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(co_occurences)
plt.show()


In [ ]:
import glob

bestfs_traces = glob.glob('../runs/bestfs-novel-2/*')
# diversity_traces = glob.glob('../runs/search/diversity-temp=1/2024_07_22/13_33_28/traces/0/*')
diversity_traces = glob.glob('../runs/search/diversity-search/2024_07_25/15_17_51/traces/0/*')



In [ ]:
names = { d.split('/')[-1] for d in bestfs_traces }
diversity_traces = [d for d in diversity_traces if d.split('/')[-1] in names]
names = { d.split('/')[-1] for d in diversity_traces }
bestfs_traces = [d for d in bestfs_traces if d.split('/')[-1] in names]
                    

In [ ]:
len(bestfs_traces)

In [ ]:
from tqdm import tqdm
import pickle

successes = []
fails = []
for i in tqdm(range(len(bestfs_traces))):
    try:
        with open(bestfs_traces[i], 'rb') as f:
            bestfs = pickle.load(f)
        with open(diversity_traces[i], 'rb') as f:
            diversity = pickle.load(f)
    except:
        continue
        
    if bestfs.proof and not diversity.proof:
        successes.append((bestfs, diversity))
    elif not bestfs.proof and diversity.proof:
        fails.append((bestfs, diversity))
        
        
        

In [ ]:
len(successes)

In [ ]:
len(fails)

In [ ]:
# [s[0].proof for s in successes]

In [ ]:
# [s[1].proof for s in fails]

In [ ]:
best_trace, diverse_trace = successes[0]


In [ ]:
best_trace.tree.data['augmented_state'] == diverse_trace.tree.data['augmented_state']

In [ ]:
set([t.tactic for t in best_trace.tree.out_edges])# == 

In [ ]:
set([t.tactic for t in diverse_trace.tree.out_edges])

In [ ]:
best_trace.proof

In [ ]:
len([d for d in best_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)])


In [ ]:
len([d for d in diverse_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)]), len([d for d in best_trace.tree.out_edges if not isinstance(d.dst[0], ErrorNode)])



In [ ]:
[(d,d.dst[0]) for d in diverse_trace.tree.out_edges ]

In [ ]:
from experiments.end_to_end.proof_node import ErrorNode

node1 = [d.dst for d in best_trace.tree.out_edges if d.tactic == best_trace.proof[0]][0][0]

In [ ]:
node2 = [d.dst for d in node1.out_edges if d.tactic == best_trace.proof[1]]
node2[0][0]

In [ ]:
diverse1  = [d.dst[0] for d in diverse_trace.tree.out_edges if d.dst[0 ] == node1][0]


In [ ]:
diverse2 = [d.dst for d in diverse1.out_edges if d.dst[0] == node2[0][0]]

In [ ]:
diverse2

In [ ]:
best_trace

In [ ]:
diverse_trace

In [ ]:
# bestfs success 0, missed out on state by removing simp tactic with relevant lemma (depth 1)
# success 1, missed out on proof at first level by removing rw tactic with relevant lemma (depth 0)
# success 2, missed out on proof at first level by removing rw tactic with relevant lemma (depth 0). Other tactics with same lemma, however they resulted in an error.
# success 3, missed tactic but had lemma (simp_rw<- vs simp_rw, timed out)
# success 4, 